In [1]:
%pip install -q lyricsgenius python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path

import pandas as pd
import lyricsgenius
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

load_dotenv(PROJECT_ROOT / '.env')

GENIUS_ACCESS_TOKEN = os.getenv('GENIUS_ACCESS_TOKEN')

if not GENIUS_ACCESS_TOKEN:
    raise ValueError('Missing GENIUS_ACCESS_TOKEN in .env')

## 1) Initialise Genius client

In [3]:
genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True
)
genius.verbose = False
print('Genius client ready.')

Genius client ready.


---
## Build per-region top songs from Spotify Charts

Reads the downloaded Spotify Charts CSVs and keeps only each region's top-ranked song.
No language detection is applied at this stage.

1. Merge all songs into single df with just `artist`, `title`, `spotify_uri`
2. Fetch lyrics into `lyrics`

Output columns: `rank`, `artist`, `title`, `region`, `spotify_uri`

In [4]:
from pathlib import Path
import pandas as pd

print('Top-song extraction helpers loaded.')

Top-song extraction helpers loaded.


In [5]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'titles.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

FILE_REGION_MAP = {
    'regional-ar-weekly-2026-03-05.csv': 'Argentina',
    'regional-es-weekly-2026-03-05.csv': 'Spain',
    'regional-jp-weekly-2026-03-05.csv': 'Japan',
    'regional-sg-weekly-2026-03-05.csv': 'Singapore',
}

def extract_date_from_filename(filename: str) -> str:
    """Extract date in YYYY-MM-DD format from filename like 'regional-XX-weekly-YYYY-MM-DD.csv'"""
    match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)
    return match.group(1) if match else None

dfs = []
for filename, region in FILE_REGION_MAP.items():
    chart_date = extract_date_from_filename(filename)
    df = pd.read_csv(RAW_DIR / filename)
    df = df.rename(columns={'artist_names': 'artist', 'track_name': 'title'})
    df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
    df = df.dropna(subset=['rank'])

    df = df.assign(
        region=region,
        chart_date=chart_date,
        spotify_uri=df['uri'].str.replace('spotify:track:', '', regex=False)
    )[["rank", "artist", "title", "region", "chart_date", "spotify_uri"]]

    dfs.append(df)

titles_df = (
    pd.concat(dfs)
    .sort_values(['region', 'rank'])
    .reset_index(drop=True)
)

titles_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to: {OUTPUT_PATH}')
print(f'Columns: {list(titles_df.columns)}')

Saved to: d:\Users\Documents\GitHub\lyrics_analysis\data\processed\titles.csv
Columns: ['rank', 'artist', 'title', 'region', 'chart_date', 'spotify_uri']


In [6]:
titles_df

,rank,artist,title,region,chart_date,spotify_uri
0,1,"Ryan Castro, Kapo, Gangsta",LA VILLA,Argentina,2026-03-05,2ZyrAym0sRLwt4PhGotHuI
1,2,Bad Bunny,BAILE INoLVIDABLE,Argentina,2026-03-05,2lTm559tuIvatlT1u0JYG2
2,3,"El Bogueto, Yung Beef",Cuando No Era Cantante,Argentina,2026-03-05,6N2iccqxRInhTLHc2Fu3W0
3,4,La T y La M,Soy Favela,Argentina,2026-03-05,3TfRpsYPQSXqqramSoWlNg
4,5,Max Carra,UWAIE - versión cumbia,Argentina,2026-03-05,6UO9DhCUq4ZbQz7PXwelFV
...,...,...,...,...,...,...
795,196,"Yapi, SOUNDPLUG",POR TI,Spain,2026-03-05,3Ip1sc72P5uz8BUB9kb4QQ
796,197,La Oreja de Van Gogh,Rosas,Spain,2026-03-05,4waqcUQWdj0yH26STWl2Rq
797,198,C. Tangana,Mala Mujer,Spain,2026-03-05,067QpM2xLhK24hTiovC3Zo
798,199,Sabrina Carpenter,Manchild,Spain,2026-03-05,2BwO5K8Q7EPAJSGze3AAh9


---
## Genius lyrics fetch from unique tracks in titles.csv

Loads titles from data/processed/titles.csv, keeps unique tracks by:
`artist`, `title`, `spotify_uri`, then fetches lyrics and writes:
`data/processed/lyrics.csv`

Output columns in lyrics.csv:
`artist`, `title`, `spotify_uri`, `lyrics`

- Uses cache file for resumability
- Skips already fetched spotify_uri values
- Writes in chunks to avoid losing progress on interruption

In [7]:
import time
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root
TITLES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'titles.csv'
CACHE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'lyrics_cache.csv'  # one row per unique track
LYRICS_OUT = PROJECT_ROOT / 'data' / 'processed' / 'lyrics.csv'        # unique tracks + lyrics

CHUNK_SIZE = 10     # flush cache every N fetched tracks
SLEEP_BETWEEN = 0.4 # seconds between Genius API calls

CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

cache_cols = ['artist', 'title', 'spotify_uri', 'lyrics']
lyrics_cols = ['artist', 'title', 'spotify_uri', 'lyrics']

titles_df = pd.read_csv(TITLES_PATH)
unique_tracks = (
    titles_df[['artist', 'title', 'spotify_uri']]
    .dropna(subset=['spotify_uri'])
    .drop_duplicates(subset=['artist', 'title', 'spotify_uri'])
    .reset_index(drop=True)
)

print(f'Loaded titles: {len(titles_df)} rows')
print(f'Unique tracks for lyrics fetch: {len(unique_tracks)} rows')

Loaded titles: 800 rows
Unique tracks for lyrics fetch: 694 rows


In [ ]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    """Return lyrics string or empty string on failure."""
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f'  [warn] {title!r} by {artist!r}: {e}')
    return ''

def append_cache_chunk(rows_chunk: list[dict], out_path: Path) -> None:
    """Append cache rows (artist, title, spotify_uri, lyrics) to lyrics_cache.csv."""
    if not rows_chunk:
        return
    chunk_df = pd.DataFrame(rows_chunk)[cache_cols]
    write_header = not out_path.exists()
    chunk_df.to_csv(out_path, mode='a', header=write_header, index=False)

# Resume: load existing cache
if CACHE_PATH.exists():
    cache_df = pd.read_csv(CACHE_PATH)
    fetched_uris = set(cache_df['spotify_uri'].astype(str))
    print(f'Cache: {len(fetched_uris)} tracks already fetched')
else:
    fetched_uris = set()
    print('No cache found, starting fresh')

# Pending tracks not yet in cache
pending = unique_tracks[~unique_tracks['spotify_uri'].astype(str).isin(fetched_uris)].reset_index(drop=True)
total = len(pending)
print(f'Tracks to fetch now: {total} (skipped {len(fetched_uris)} cached)')

# Fetch loop
if total == 0:
    print('All tracks already cached, skipping fetch.')
else:
    rows_buffer = []
    for i, row in enumerate(pending.itertuples(), 1):
        lyrics = fetch_lyrics_genius(genius, row.title, row.artist)
        rows_buffer.append({
            'artist': row.artist,
            'title': row.title,
            'spotify_uri': row.spotify_uri,
            'lyrics': lyrics,
        })

        status = 'ok' if lyrics else 'missing'
        print(f'[{i}/{total}] {status} | {row.title[:60]}')

        if len(rows_buffer) >= CHUNK_SIZE:
            append_cache_chunk(rows_buffer, CACHE_PATH)
            print(f'  -> flushed {len(rows_buffer)} rows to cache')
            rows_buffer = []

        time.sleep(SLEEP_BETWEEN)

    if rows_buffer:
        append_cache_chunk(rows_buffer, CACHE_PATH)
        print(f'  -> flushed final {len(rows_buffer)} rows to cache')

# Finalize lyrics.csv with required schema
cache_df = pd.read_csv(CACHE_PATH)
lyrics_df = (
    cache_df[lyrics_cols]
    .drop_duplicates(subset=['artist', 'title', 'spotify_uri'])
    .reset_index(drop=True)
)
lyrics_df.to_csv(LYRICS_OUT, index=False)
print(f'Wrote {len(lyrics_df)} rows to {LYRICS_OUT}')
print(f'Columns: {list(lyrics_df.columns)}')

No cache found, starting fresh
Tracks to fetch now: 694 (skipped 0 cached)
[1/694] ok | LA VILLA
[2/694] ok | BAILE INoLVIDABLE
[3/694] ok | Cuando No Era Cantante
[4/694] ok | Soy Favela
[5/694] missing | UWAIE - versión cumbia
[6/694] ok | Puñaladas
[7/694] ok | DtMF
[8/694] ok | Tu jardín con enanitos
[9/694] ok | VOY A LLeVARTE PA PR
[10/694] ok | Cuando No Era Cantante - Remix
  -> flushed 10 rows to cache
[11/694] ok | FOREVER TU GANTEL
[12/694] ok | TU VAS SIN (fav)
[13/694] ok | EoO
[14/694] ok | VeLDÁ
[15/694] ok | La Perla
[16/694] ok | NUEVAYoL
[17/694] ok | no tiene sentido
[18/694] ok | Amor de Vago
[19/694] ok | Pensamientos
[20/694] ok | La Plena - W Sound 05
  -> flushed 10 rows to cache
[21/694] ok | Niño
[22/694] ok | JETSKI - Remix
[23/694] ok | No Se Va
[24/694] ok | mi refe
[25/694] ok | YO y TÚ
[26/694] ok | SOLEAO
[27/694] ok | capaz (merengueton)
[28/694] ok | KLOuFRENS
[29/694] ok | Qué Pasaría...
[30/694] ok | LA CANCIÓN
  -> flushed 10 rows to cache
[31/694] 

In [ ]:
lyrics_df = pd.read_csv(LYRICS_OUT)

print(f'Saved rows: {len(lyrics_df)}')
print(f'Output path: {LYRICS_OUT}')
print(f'Columns: {list(lyrics_df.columns)}')

coverage = (
    lyrics_df
    .assign(has_lyrics=lyrics_df['lyrics'].fillna('').str.strip().ne(''))['has_lyrics']
    .value_counts()
)

print('\nLyrics availability:')
print(coverage.to_string())

lyrics_df.head(10)